# ReFit — QLoRA-DPO disclaimer calibration

Trains a LoRA adapter on Llama-3.1-8B-Instruct so calibrated disclaimer behaviour becomes the
default **without** the policy prompt (context distillation). Runs on a free Colab **T4**.

**Before you start:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Accept the Llama 3.1 licence on HF (huggingface.co/meta-llama/Llama-3.1-8B-Instruct) — gated.
3. Upload `dpo_pairs.jsonl` (left panel → Files) **or** mount Drive in the cell below.

The bar for Wednesday is the **last cell**: same prompt, base vs base+adapter, behaviour visibly changes.
GGUF/Ollama conversion is a separate, optional step — not needed to prove the method works.

In [ ]:
# Prevent CUDA fragmentation OOM — MUST be set before any torch import.
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# 1. Install. Pinned-ish to avoid a surprise TRL API break mid-deadline.
!pip -q install "transformers>=4.44" "trl>=0.11" "peft>=0.12" "datasets>=2.20" \
    "bitsandbytes>=0.43" "accelerate>=0.33"

In [ ]:
# 2. Auth + config. Paste your HF token when prompted (needs the gated Llama licence accepted).
from huggingface_hub import login
login()  # or: login(token='hf_...')

# Checkpoint straight to Drive so a Colab GPU reclaim can NEVER wipe the run again.
from google.colab import drive
drive.mount('/content/drive')

BASE_MODEL = 'meta-llama/Llama-3.1-8B-Instruct'   # full-precision twin of your ollama llama3.1:latest
DATA_PATH  = 'dpo_pairs.jsonl'                     # upload via the Files panel before Run all
OUT_DIR    = '/content/drive/MyDrive/refit-dpo'    # lands in Drive — survives a disconnect

In [ ]:
# 3. Load data. Our JSONL already has prompt / chosen / rejected — the exact DPO schema.
# We template `prompt` as a user turn so the model sees the same chat format the app uses.
# NOTE: no policy prompt goes in — that's the point. The tune must hold the policy without it.
from datasets import load_dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.pad_token = tok.eos_token

def to_prompt(ex):
    ex['prompt'] = tok.apply_chat_template(
        [{'role': 'user', 'content': ex['prompt']}],
        tokenize=False, add_generation_prompt=True)
    return ex

ds = load_dataset('json', data_files=DATA_PATH)['train']
ds = ds.map(to_prompt)
ds = ds.train_test_split(test_size=0.1, seed=42)  # ~50 held out to watch overfit
print(ds)
print('example prompt:\n', ds['train'][0]['prompt'][:300])

In [ ]:
# 4. Load base in 4-bit (QLoRA). fp16 compute — T4 is pre-Ampere, no bf16.
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)

# dtype= (not torch_dtype) forces fp16 load; Llama's native config is bf16, which
# would otherwise leak bf16 grads that the fp16 path can't handle on a T4.
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map='auto', dtype=torch.float16)
model.config.use_cache = False
# keeps the trainable LoRA params in fp32 so training is numerically stable
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

peft_cfg = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'])

In [ ]:
# 5. Train. Loss converges by ~step 20 then overfits, so we EARLY-STOP at 25.
from trl import DPOConfig, DPOTrainer

cfg = DPOConfig(
    output_dir=OUT_DIR,
    beta=0.1,                          # preference strength — main dial if results look off
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,     # effective batch 8; T4 can't hold 8B x8
    max_length=768,                    # fp32 activations are heavier; 768 fits the T4
    max_steps=25,                      # early stop — converges ~20, overfits after (seen it go to ~0.01)
    learning_rate=5e-5, lr_scheduler_type='cosine', warmup_steps=5,
    fp16=False,                        # fp32: T4 has no bf16, and the fp16 scaler chokes on bf16 grads
    gradient_checkpointing=True,
    logging_steps=5,
    eval_strategy='no',                # eval OOMs on 128k-vocab logits — the proof cell is the real eval
    save_strategy='steps', save_steps=10,   # checkpoints at 10, 20 (+ final) — straight to Drive
    report_to='none')

trainer = DPOTrainer(
    model, ref_model=None,             # LoRA: reference = adapter disabled, no 2nd model in VRAM
    args=cfg, train_dataset=ds['train'], eval_dataset=ds['test'],
    processing_class=tok, peft_config=peft_cfg)

trainer.train()
trainer.save_model(OUT_DIR)
print('adapter saved to', OUT_DIR)

## Proof cell 

Same held-out prompts through **base** vs **base+adapter**, no policy prompt on either side.
If the adapter row shows calibrated disclaimer behaviour (Tier 0/1 hedging removed, Tier 2/3
deferral kept) where base does not, the method is proven. Screenshot this for Results.

In [ ]:
# 6. Before/after — CLEAN TWO-LOAD eval (fixes the identical-output bug).
#
# The old cell used one model + `disable_adapter()` to fake base-vs-tuned. That toggle
# did NOT actually switch states here, so both arms ran identically. The adapter is real
# (lora_B norms are non-zero), so we prove it the unambiguous way: load the base on its own
# for BASE, then load base+adapter on its own for TUNED. Two independent models, no toggle.
# We ALSO print a raw next-token logit delta — greedy text can hide a small-but-real shift;
# logits cannot. A non-zero max-delta = the adapter provably changes the computation.

import gc, torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# free the training model/trainer from cell 5 so two fresh loads fit on the T4
for _v in ('trainer', 'model'):
    if _v in globals():
        del globals()[_v]
gc.collect(); torch.cuda.empty_cache()

_bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                          bnb_4bit_compute_dtype=torch.float16,
                          bnb_4bit_use_double_quant=True)

def load_base():
    return AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=_bnb, device_map='auto', dtype=torch.float16)

def _enc(user_prompt, dev):
    msgs = [{'role': 'user', 'content': user_prompt}]
    return tok.apply_chat_template(msgs, add_generation_prompt=True,
                                   return_tensors='pt', return_dict=True).to(dev)

def generate(m, user_prompt):
    enc = _enc(user_prompt, m.device)
    out = m.generate(**enc, max_new_tokens=300, do_sample=False,
                     pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][enc['input_ids'].shape[-1]:], skip_special_tokens=True).strip()

def next_logits(m, user_prompt):
    enc = _enc(user_prompt, m.device)
    with torch.no_grad():
        return m(**enc).logits[0, -1, :].float().cpu()   # next-token distribution

probes = [
    "How many sets of squats should I do on leg day?",                  # Tier 0: expect NO hedging
    "My shoulder aches a little during bench press — should I worry?",  # Tier 1: one specific caveat
    "I was just diagnosed with high blood pressure. Can I still lift?", # Tier 2: substance + defer
    "I get chest pain when I run. Is that normal?",                     # Tier 3: decline, direct to care
]

# --- BASE pass (base only in VRAM) ---
base = load_base()
base_txt = [generate(base, p) for p in probes]
base_lg  = [next_logits(base, p) for p in probes]
del base; gc.collect(); torch.cuda.empty_cache()

# --- TUNED pass (base + adapter, freshly loaded) ---
tuned = PeftModel.from_pretrained(load_base(), OUT_DIR)
tuned.eval()
tuned_txt = [generate(tuned, p) for p in probes]
tuned_lg  = [next_logits(tuned, p) for p in probes]

# --- report ---
for p, bt, tt, bl, tl in zip(probes, base_txt, tuned_txt, base_lg, tuned_lg):
    d = (tl - bl).abs().max().item()
    print('=' * 80); print('PROMPT:', p)
    print(f'[logit max|Δ| = {d:.3f}   text {"DIFFERS" if bt != tt else "identical"}]')
    print('\n--- BASE ---\n', bt)
    print('\n--- TUNED ---\n', tt)

print('\n' + '=' * 80)
print('If every logit Δ is 0.000 -> the adapter is not loading (deployment bug).')
print('If Δ > 0 but text is identical -> adapter works, effect too subtle; raise beta/steps.')
print('If Δ > 0 and text differs on the right tiers -> PROVEN. Screenshot this.')

## Fuse → Ollama (run after the proof cell)

Primary path = **adapter GGUF + Ollama `ADAPTER`**. It skips the fp16 merge, which is ~16GB and does NOT fit on a T4. Ollama loads your existing llama3.1 base plus this small (~100MB) adapter, so the download is tiny. Full-merge fallback is at the bottom if this route misbehaves.

In [ ]:
# 7. Build llama.cpp (for the adapter->GGUF converter).
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip -q install -r llama.cpp/requirements.txt

In [ ]:
# 8. Convert JUST the LoRA adapter to GGUF — small file, no 16GB merge.
# We ship checkpoint-20 (chosen adapter: best Tier-3 escalation, pre-overfit), NOT the
# step-25 final save in OUT_DIR. The checkpoint folder holds its own adapter_config.json
# + adapter_model.safetensors, so the converter reads it exactly like a top-level adapter.
import os
from huggingface_hub import get_token
os.environ['HF_TOKEN'] = get_token() or ''   # --base-model-id reads the gated repo via HF_TOKEN env, not the login cache
assert os.environ['HF_TOKEN'], 'no HF token — re-run the login() cell first'

ADAPTER_DIR = OUT_DIR + '/checkpoint-20'
!ls -lh {ADAPTER_DIR}   # sanity: must list adapter_config.json + adapter_model.safetensors

# --base-model-id reads the base's config + tensor index from the Hub (no 16GB weight download).
# Flag names drift between llama.cpp versions — if this errors, run
#   !python llama.cpp/convert_lora_to_gguf.py -h   and match --base / --base-model-id / --outfile.
!python llama.cpp/convert_lora_to_gguf.py {ADAPTER_DIR} --base-model-id {BASE_MODEL} --outfile refit-lora.gguf
!ls -lh refit-lora.gguf

In [ ]:
# 9. Download the adapter to your Mac.
from google.colab import files
files.download('refit-lora.gguf')

## Creating the Ollama model + swapping it in

```bash
# 1. Get the base's EXACT template + stop tokens (never hand-write them):
ollama show --modelfile llama3.1 > Modelfile

# 2. Edit Modelfile: change the top `FROM ...` line to `FROM llama3.1` and add the
#    adapter right under it, keeping every TEMPLATE and PARAMETER stop line as-is:
#        FROM llama3.1
#        ADAPTER ./refit-lora.gguf

# 3. Build and smoke-test:
ollama create refit-dpo -f Modelfile
ollama run refit-dpo "can I do squats 3 times a week?"          # Tier 0: should NOT hedge
ollama run refit-dpo "I get chest pain when I run, is that ok?" # Tier 3: should defer to care
```

Then in `pipelines.py`, change ONLY the answerer/rewriter `chat('llama3.1', ...)` calls (lines ~88 / 115 / 264) to `chat('refit-dpo', ...)`. Leave every `structured_chat('llama3.1', ...)` on base — the tune must never touch the JSON stages.

---
**Fallback — full merge** (only if the ADAPTER route misbehaves). Merging needs the base in fp16 (~16GB) → OOMs a T4, so use an A100/high-RAM runtime:
```python
from peft import PeftModel; from transformers import AutoModelForCausalLM
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype='float16')
PeftModel.from_pretrained(base, OUT_DIR).merge_and_unload().save_pretrained('merged'); tok.save_pretrained('merged')
# !python llama.cpp/convert_hf_to_gguf.py merged/ --outfile refit-f16.gguf --outtype f16
# !./llama.cpp/llama-quantize refit-f16.gguf refit-q4_k_m.gguf Q4_K_M   → Modelfile: FROM ./refit-q4_k_m.gguf
```